# IMPORTS

In [1]:
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, cross_val_score, StratifiedKFold

In [2]:
import os
from google.colab import drive
drive.mount('/content/gdrive')

# Change working directory to be current folder
# os.chdir('/content/gdrive/My Drive/Your Folder Name/Your sub Folder Name')
os.chdir('/content/gdrive/My Drive/Projects_IAN/iris_production')
!ls

Mounted at /content/gdrive
iris.csv	   iris_training_logistic_regression.ipynb
iris_to_csv.ipynb  iris_training_XGBoost.ipynb


# Preprocess, Training and Test Set

In [3]:
df = pd.read_csv("iris.csv")

In [4]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2,virginica
146,6.3,2.5,5.0,1.9,2,virginica
147,6.5,3.0,5.2,2.0,2,virginica
148,6.2,3.4,5.4,2.3,2,virginica


In [5]:
X, y = df.iloc[:, :-2], df.iloc[:, -2]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression

In [7]:
model = model = XGBClassifier()
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

# Evaluation on Simple Train Test

In [8]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

In [9]:
print(accuracy)
print(classification_report(y_test, y_pred))

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [10]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


# Evaluation and Fit with K_Fold


In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

# scores = cross_val_score(log_reg_model, X, y, cv=skf)

scores = cross_validate(
    model,
    X,
    y,
    cv=skf,
    scoring=metrics
)

# print(scores.mean())
print(scores)


{'fit_time': array([0.03502893, 0.03231835, 0.03074002, 0.0318315 , 0.03109765]), 'score_time': array([0.01234889, 0.01175523, 0.01175857, 0.01123118, 0.01174545]), 'test_accuracy': array([0.96666667, 0.96666667, 0.9       , 0.96666667, 0.9       ]), 'test_precision_macro': array([0.96969697, 0.96969697, 0.91414141, 0.96969697, 0.9023569 ]), 'test_recall_macro': array([0.96666667, 0.96666667, 0.9       , 0.96666667, 0.9       ]), 'test_f1_macro': array([0.96658312, 0.96658312, 0.89500042, 0.96658312, 0.89974937])}


In [12]:
accuracy = scores["test_accuracy"]
precision = scores["test_precision_macro"]
recall = scores["test_recall_macro"]
f1 = scores["test_f1_macro"]

In [13]:
print(f"accuracy =    {accuracy}")
print(f"percision =   {precision}")
print(f"recall =      {recall}")
print(f"f1 =          {f1}")

accuracy =    [0.96666667 0.96666667 0.9        0.96666667 0.9       ]
percision =   [0.96969697 0.96969697 0.91414141 0.96969697 0.9023569 ]
recall =      [0.96666667 0.96666667 0.9        0.96666667 0.9       ]
f1 =          [0.96658312 0.96658312 0.89500042 0.96658312 0.89974937]


# Evaluation and Fit with K_Fold (explicit way)


In [15]:
scores = []

for train_index, test_index in skf.split(X,y):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    score = accuracy_score(y_test, preds)

    scores.append(score)

print(np.mean(scores)) # scores here is list not numpy array
print(scores)

0.9400000000000001
[0.9666666666666667, 0.9666666666666667, 0.9, 0.9666666666666667, 0.9]


# Testing Syntax


In [16]:
[y_test.iloc[0:3]]

[3     0
 5     0
 16    0
 Name: target, dtype: int64]

In [17]:
model.predict(X_test.iloc[[0]])

array([0])

In [18]:
model.predict_proba(X_test.iloc[[0]])[0][model.predict(X_test.iloc[[0]])]

array([0.9930351], dtype=float32)

In [19]:
probs = model.predict_proba(X_test)
print(probs[:3])

[[0.9930351  0.0057217  0.00124329]
 [0.9930351  0.0057217  0.00124329]
 [0.9930351  0.0057217  0.00124329]]
